# 03 · Frame extraction

Samples a fixed number of frames from every labelled angiographic series,
converts them to 224 x 224 greyscale PNGs and writes an index
(`frames_index.csv`) that the datasets read.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "angiocad_common.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import angiocad_common as ac
import json
import re

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ac.ensure_dirs()
ac.FRAMES_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (ac.IMG_SIZE, ac.IMG_SIZE)
MAX_FRAMES_PER_SERIES = 10
# Fraction of each series to skip at both ends before sampling. The first and
# last frames of a run often show little contrast. 0 keeps the whole run;
# 0.1 is a reasonable setting if you want to avoid those frames.
TRIM_FRACTION = 0.0
IMAGE_EXT = {".jpg", ".jpeg", ".png"}

In [3]:
def natural_key(path: Path):
    """Sort frame_2 before frame_10 even when the numbers are not zero-padded."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", path.name)]


def list_series_files(series_dir: Path) -> list[Path]:
    """All frames of one series in acquisition order."""
    return sorted((f for f in series_dir.iterdir() if f.suffix.lower() in IMAGE_EXT), key=natural_key)


def sample_frames(files: list[Path], k: int, trim: float = 0.0) -> list[Path]:
    """Pick k frames evenly spaced over the (optionally trimmed) series."""
    cut = int(len(files) * trim)
    if cut and len(files) - 2 * cut >= k:
        files = files[cut:len(files) - cut]
    if len(files) <= k:
        return files
    idx = np.linspace(0, len(files) - 1, k).round().astype(int)
    return [files[i] for i in idx]


def load_gray(path: Path, size: tuple[int, int]):
    """Read through numpy so paths with non-ASCII characters work on Windows."""
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    return None if img is None else cv2.resize(img, size, interpolation=cv2.INTER_AREA)


def save_png(path: Path, img) -> bool:
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(str(path))
    return bool(ok)


def split_series(value) -> list[str]:
    """'1-3-5' -> ['1', '3', '5']"""
    return [] if pd.isna(value) else [t.strip() for t in str(value).split("-") if t.strip()]

In [4]:
cohort = pd.read_csv(ac.OUTPUT_DIR / "clinical_features_original.csv")
print(f"Patients in the cohort: {len(cohort)}")

records, unreadable, missing_series, patients_without_folder = [], [], [], []

for _, row in tqdm(cohort.iterrows(), total=len(cohort), desc="Extracting"):
    pid = int(row["ID"])
    patient_dir = ac.PATIENT_DIR / str(pid)
    if not patient_dir.is_dir():
        patients_without_folder.append(pid)
        continue

    for side, col in (("R", "Right Coronary Series"), ("L", "Left Coronary Series")):
        for series_id in split_series(row[col]):
            series_dir = patient_dir / series_id
            files = list_series_files(series_dir) if series_dir.is_dir() else []
            if not files:
                missing_series.append(f"{pid}/{series_id}")
                continue

            for idx, src in enumerate(sample_frames(files, MAX_FRAMES_PER_SERIES, TRIM_FRACTION)):
                img = load_gray(src, IMG_SIZE)
                out_path = ac.FRAMES_DIR / f"{pid}_{side}_{series_id}_{idx:02d}.png"
                if img is None or not save_png(out_path, img):
                    unreadable.append(str(src))
                    continue
                records.append({"ID": pid, "side": side, "series": series_id, "frame_idx": idx,
                                "n_frames_in_series": len(files),
                                "source": ac.relative_path(src),
                                "path": ac.relative_path(out_path)})

frames = pd.DataFrame(records)
frames.to_csv(ac.OUTPUT_DIR / "frames_index.csv", index=False)

Patients in the cohort: 377


Extracting:   0%|          | 0/377 [00:00<?, ?it/s]

In [5]:
# PNGs from an earlier run that are no longer in the index. They are harmless
# (only indexed frames are ever read) but can be deleted by hand to save space.
indexed = {Path(p).name for p in frames["path"]}
stale = [f for f in ac.FRAMES_DIR.glob("*.png") if f.name not in indexed]

In [6]:
with_frames = set(frames["ID"])
without_frames = sorted(set(cohort["ID"].astype(int)) - with_frames)

report = {
    "n_patients": int(len(cohort)),
    "n_patients_with_frames": len(with_frames),
    "patients_without_frames": without_frames,
    "patients_without_folder": patients_without_folder,
    "n_frames": int(len(frames)),
    "frames_by_side": frames["side"].value_counts().to_dict(),
    "missing_or_empty_series": missing_series,
    "unreadable_files": unreadable,
    "max_frames_per_series": MAX_FRAMES_PER_SERIES,
    "trim_fraction": TRIM_FRACTION,
    "image_size": list(IMG_SIZE),
}
with open(ac.OUTPUT_DIR / "frame_extraction_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"Frames written           : {len(frames):,}")
print(f"  right / left           : {report['frames_by_side']}")
print(f"Patients with frames     : {len(with_frames)} of {len(cohort)}")
print(f"Patients without frames  : {without_frames}")
print(f"Missing or empty series  : {len(missing_series)}")
print(f"Unreadable files         : {len(unreadable)}")
print(f"PNGs not in the index    : {len(stale)}")

Frames written           : 13,353
  right / left           : {'L': 7137, 'R': 6216}
Patients with frames     : 376 of 377
Patients without frames  : [157]
Missing or empty series  : 42
Unreadable files         : 0
PNGs not in the index    : 0


In [7]:
# Every indexed file exists, and every frame belongs to a cohort patient.
not_on_disk = [p for p in frames["path"] if not Path(ac.resolve_path(p)).is_file()]
assert not not_on_disk, f"{len(not_on_disk)} indexed frames are missing, e.g. {not_on_disk[:3]}"
assert with_frames <= set(cohort["ID"].astype(int))

per_patient = frames.groupby("ID").size()
print(f"Frames per patient: median {per_patient.median():.0f}, "
      f"range {per_patient.min()}-{per_patient.max()}")

Frames per patient: median 40, range 10-40
